In [0]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "m5")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
spark.sql(f"USE {CATALOG}.{SCHEMA}")
RAW = f"/Volumes/{CATALOG}/{SCHEMA}/raw"

In [0]:
from pyspark.sql import DataFrame, functions as F

ROW_RULES = {
    "units_not_null":      "units IS NOT NULL",
    "units_non_negative":  "units >= 0",
    "date_matched":        "date IS NOT NULL",
    "price_positive":      "sell_price > 0",
}

def apply_row_rules(df: DataFrame, rules: dict = ROW_RULES):
    failed = [F.when(~F.coalesce(F.expr(cond), F.lit(False)), F.lit(name))
              for name, cond in rules.items()]
    tagged = df.withColumn("_failed_rules", F.array_compact(F.array(*failed)))
    valid = tagged.filter(F.size("_failed_rules") == 0).drop("_failed_rules")
    quarantine = tagged.filter(F.size("_failed_rules") > 0)
    return valid, quarantine

def duplicate_keys(df: DataFrame, keys: list) -> int:
    return df.groupBy(*keys).count().filter("count > 1").count()